# STEAM-GAME-PROJECT — Phân tích dữ liệu Non-ML

## Mục tiêu và phạm vi

Khung mẫu cho **Giai đoạn 3.3 — Xây dựng câu hỏi nghiên cứu** và **Giai đoạn 3.4 — Phân tích dữ liệu Non-ML** của 5 câu hỏi Non-ML. Chưa thực hiện phân tích và chưa có kết quả nghiên cứu; nội dung 3.3 là kế hoạch, còn 3.4 là khung triển khai.

## Tập dữ liệu

- **Tập dữ liệu chính:** Kaggle Steam Games, bản tháng 3 năm 2025; tệp hiện có `data/raw/steam_games_mar2025.csv`.
- **Tập dữ liệu bổ sung:** `data/raw/hf_steam_games_dataset_train.parquet` từ Hugging Face; chỉ được xem xét cho RQ1 sau khi xác minh thuộc tính bổ sung có giá trị, khóa và mốc mốc snapshot tương thích.
- README và mặc định trong `src/data_loader.py` nêu `games_march2025_cleaned.csv`, khác tên CSV hiện có. Khi triển khai, truyền đường dẫn tương đối rõ ràng. `src/preprocessing.py` hiện chưa có.
- Notebook 01 đã thực hiện EDA chung. Tái sử dụng quy tắc phù hợp, không lặp lại toàn bộ EDA; xác minh giá trị sentinel, số 0, đơn vị đo và nguồn của từng biến.

## Năm câu hỏi Non-ML chính thức

Theo bảng phân công trong CONTRIBUTIONS, Q3 là câu hỏi Machine Learning thuộc Notebook 03; notebook này gồm Q1, Q2, Q4, Q5 và Q6.

1. RQ1 (Q1 nhóm): Mở rộng Độ phủ & Quy mô Người chơi.
2. RQ2 (Q2 nhóm): Phân hóa Thời lượng chơi (Playtime Distribution).
3. RQ3 (Q4 nhóm): Giá trị Nội dung Bổ sung (DLC & Achievements).
4. RQ4 (Q5 nhóm): Động lực Tương tác F2P vs Pay-to-Play.
5. RQ5 (Q6 nhóm): Nghịch lý Thời lượng chơi (The Playtime Paradox).

Câu hỏi được trích nguyên văn; 3.3 là kế hoạch, 3.4 là phần triển khai sau này.


# 3.3. Xây dựng câu hỏi nghiên cứu

Project brief yêu cầu mỗi câu hỏi có ý nghĩa và nêu lợi ích. D1–D6 dưới đây là kế hoạch, không phải kết quả đã hoàn thành; chỉ dùng tiêu chí phù hợp.

## 3.3.1. Câu hỏi nghiên cứu 1: Mở rộng Độ phủ & Quy mô Người chơi (Q1 nhóm)

### 1. Câu hỏi nghiên cứu

> “Số lượng ngôn ngữ và hệ điều hành được hỗ trợ có mối liên hệ như thế nào với lượng người sở hữu ước tính, và mối liên hệ này thay đổi ra sao khi xét đến thể loại và tuổi đời game?”

Mối liên hệ ở cấp game giữa độ phủ ngôn ngữ/nền tảng và số người sở hữu ước tính, xem xét `thể loạis`, tuổi đời; không suy luận nhân quả.

### 2. Động cơ nghiên cứu

Nhà phát triển/nhà phát hành cần hiểu độ phủ ngôn ngữ và nền tảng liên hệ thế nào với quy mô sở hữu theo phân khúc để ưu tiên thị trường, hỗ trợ nền tảng. Kết quả chỉ hỗ trợ quyết định và giả thuyết tiếp theo, không chứng minh quan hệ nhân quả.

**Các bên quan tâm:** Nhà phát triển, nhà phát hành, đơn vị phân tích game và người chơi.

**Giá trị thực tiễn:** Ưu tiên nghiên cứu/phân khúc bản địa hóa và hỗ trợ nền tảng.

### 3. Thuộc tính sử dụng

| Thuộc tính | Nguồn dữ liệu | Vai trò trong phân tích |
| --- | --- | --- |
| appid | Kaggle CSV | Khóa game/ghép. |
| supported_languages | Kaggle CSV | Tạo `language_count`. |
| os | Kaggle CSV | Danh sách hệ điều hành để tạo số nền tảng. |
| estimated_người sở hữus | Kaggle CSV | Kết quả dạng khoảng. |
| thể loạis, release_date, price | Kaggle CSV | Nhóm tuổi/biến điều chỉnh. |
| appID, khoảng thời gians, mac, linux | HF Parquet | Khóa/cờ đối chiếu; xác minh thời điểm trước ghép. |

### 4. Giả thuyết nghiên cứu

- H1: `language_count` và `platform_count` có mối liên hệ với `người sở hữu_midpoint`; không diễn giải nhân quả.
- H2: Mức mối liên hệ khác nhau theo `thể loạis`.
- H3: Mức mối liên hệ thay đổi theo `game_age` sau khi xem xét `price` và `thể loạis`.

### 5. Tiêu chí chiều sâu phân tích

Các phương pháp dưới đây mới là kế hoạch. Chỉ ghi nhận tiêu chí đã thực hiện khi có kiểm tra và kết quả tương ứng.

| Tiêu chí | Phương pháp dự kiến | Mục tiêu |
| --- | --- | --- |
| D1 — Kết hợp dữ liệu | Đối chiếu `os` với cờ nền tảng HF sau kiểm tra khóa/mốc snapshot/sai khác. | Đánh giá thông tin đối chiếu mới; chưa mặc định gộp. |
| D2 — Tái cấu trúc / Thời gian | `explode()` ngôn ngữ; tính tuổi từ mốc mốc snapshot đã xác nhận. | Tổng hợp ngôn ngữ/tuổi mà không nhân bản đơn vị kiểm định. |
| D3 — Biến dẫn xuất | `language_count`, `platform_count`, `người sở hữu_midpoint`, `game_age`. | Định nghĩa biến tái lập. |
| D4 — Kiểm soát yếu tố gây nhiễu | Phân tầng/điều chỉnh `thể loạis`, `price`, `game_age`. | Mối liên hệ có điều kiện, không loại mọi nhiễu. |
| D5 — Định lượng độ bất định | Spearman và Bootstrap 95% CI theo game; báo effect size/N. | Định lượng bất định. |
| D6 — Kiểm tra độ bền vững | Biên người sở hữu thấp/cao/trung điểm, nhóm tuổi, quy tắc giá trị ngoại lệ. | Độ nhạy với giả định. |

### 6. Giới hạn nghiên cứu

Owner là khoảng; thuộc tính hỗ trợ là khai báo; thể loạis có thể đa nhãn; dữ liệu quan sát không hỗ trợ nhân quả; mốc snapshot/khóa cần xác minh.

## 3.3.2. Câu hỏi nghiên cứu 2: Phân hóa Thời lượng chơi (Playtime Distribution) (Q2 nhóm)

### 1. Câu hỏi nghiên cứu

> “Sự chênh lệch giữa thời gian chơi trung bình và trung vị phản ánh những đặc điểm khác biệt nào trong phân phối thời lượng chơi của các tựa game, và mức độ chênh lệch này có khác nhau giữa các thể loại game hay không?”

So sánh hai thống kê tổng hợp ghép cặp ở cấp game; không tái dựng toàn bộ phân phối người chơi.

### 2. Động cơ nghiên cứu

Mean nhạy với giá trị lớn hơn trung vị (median). Đo chênh lệch theo thể loại giúp chọn cách tóm tắt mức độ tương tác và nhóm cần khảo sát thêm; không mô tả từng người chơi.

**Các bên quan tâm:** Nhà phát triển, nhà phát hành, phân tích sản phẩm và người chơi.

**Giá trị thực tiễn:** Định lượng hướng/độ lớn chênh lệch, khác biệt thể loại và độ bất định.

### 3. Thuộc tính sử dụng

| Thuộc tính | Nguồn dữ liệu | Vai trò trong phân tích |
| --- | --- | --- |
| appid, name | Kaggle CSV | Khóa/nhận diện. |
| average_playtime_trọn đời, trung vị (median)_playtime_trọn đời | Kaggle CSV | Hai số đo chính. |

RQ2 chỉ sử dụng tập dữ liệu chính; không áp dụng D1 và không dùng tập dữ liệu bổ sung.
| thể loạis, estimated_người sở hữus, release_date | Kaggle CSV | Nhóm/biến điều chỉnh. |

### 4. Giả thuyết nghiên cứu

- H1: Trong game có hai số đo dương, khác nhau, tỷ lệ average > trung vị (median) khác 50%.
- H2: Tỷ lệ chênh lệch khác 0 và độ lớn chênh lệch tuyệt đối khác nhau theo thể loạis.
- H3: Khác biệt thể loại còn sau khi xem xét người sở hữu và tuổi game.

### 5. Tiêu chí chiều sâu phân tích

Các phương pháp dưới đây mới là kế hoạch. Chỉ ghi nhận tiêu chí đã thực hiện khi có kiểm tra và kết quả tương ứng.

| Tiêu chí | Phương pháp dự kiến | Mục tiêu |
| --- | --- | --- |
| D2 — Tái cấu trúc / Thời gian | `melt()` average/trung vị (median) thành dạng long nếu cần trình bày; giữ `appid` làm khóa cặp và chỉ so sánh trọn đời. | So sánh có cấu trúc, không coi hai giá trị của cùng game là quan sát độc lập. |
| D3 — Biến dẫn xuất | Signed/absolute chênh lệch, ratio khi trung vị (median)>0, chênh lệch thứ hạng nếu phù hợp. | Phân biệt hướng/độ lớn; minh bạch không xác định. |
| D4 — Kiểm soát yếu tố gây nhiễu | Phân tầng/điều chỉnh thể loại, người sở hữu, age. | Giảm nhiễu quan sát được. |
| D5 — Định lượng độ bất định | Bootstrap 95% CI theo game; sign/group test, effect size phù hợp. | Bất định có xét nhiều số 0. |
| D6 — Kiểm tra độ bền vững | Toàn bộ so với cặp dương; chênh lệch/ratio/rank và giá trị ngoại lệ. | Độ nhạy số 0-inflation/thước đo. |

### 6. Giới hạn nghiên cứu

Playtime tổng hợp cấp game; tỷ lệ số 0 cao; đơn vị đo/nguồn/cực trị cần xác minh; chênh lệch không mô tả đầy đủ phân phối.


## 3.3.3. Câu hỏi nghiên cứu 3: Giá trị Nội dung Bổ sung (DLC & Achievements) (Q4 nhóm)

### 1. Câu hỏi nghiên cứu

> “Mật độ phân bổ thành tựu trên mỗi gói mở rộng (achievements/dlc_count) có chiều hướng tác động như thế nào đến tỷ lệ đánh giá tích cực gần đây của người chơi (pct_pos_recent), sau khi đã kiểm soát yếu tố chất lượng chuyên môn của trò chơi (metacritic_score)?”

Mối liên hệ achievements trên DLC với pct_pos_recent, điều chỉnh metacritic_score; không nhân quả.

### 2. Động cơ nghiên cứu

Nhà phát triển có thể quan tâm liệu mật độ thành tựu liên hệ với positive đánh giá gần đây sau khi xét điểm chuyên môn, để đánh giá chỉ số thiết kế cần nghiên cứu. Dữ liệu quan sát chỉ hỗ trợ mối liên hệ, không chứng minh tác động.

**Các bên quan tâm:** Nhà phát triển/nhà phát hành, thiết kế game và phân tích chất lượng.

**Giá trị thực tiễn:** Ước lượng mối liên hệ có điều kiện và đánh giá cách đo mật độ DLC/achievement.

### 3. Thuộc tính sử dụng

| Thuộc tính | Nguồn dữ liệu | Vai trò trong phân tích |
| --- | --- | --- |
| appid, achievements, dlc_count | Kaggle CSV | Khóa, tử số và mẫu số. |
| pct_pos_recent, num_đánh giás_recent | Kaggle CSV | Kết quả và độ tin cậy đánh giá. |
| metacritic_score | Không có trong schema CSV chính đã kiểm tra | Thuộc tính được nêu trong RQ nhưng chưa có nguồn được phép xác nhận; cần nhóm quyết định cách xử lý trước phân tích. |

### 4. Giả thuyết nghiên cứu

- H1: Sau điều chỉnh metacritic_score, achievements-per-DLC liên hệ với pct_pos_recent.
- H2: Mối liên hệ có thể khác theo metacritic_score hoặc nhóm DLC.
- H3 (nếu phù hợp): Kết luận ổn định qua cách xử lý DLC=0 và ngưỡng đánh giá.

### 5. Tiêu chí chiều sâu phân tích

Các phương pháp dưới đây mới là kế hoạch. Chỉ ghi nhận tiêu chí đã thực hiện khi có kiểm tra và kết quả tương ứng.

| Tiêu chí | Phương pháp dự kiến | Mục tiêu |
| --- | --- | --- |
| D3 — Biến dẫn xuất | Tạo achievements/dlc_count; tách DLC=0, không chia 0. | Thước đo mật độ tái lập. |
| D4 — Kiểm soát yếu tố gây nhiễu | Điều chỉnh metacritic_score; cân nhắc đánh giá count/thể loại. | Mối liên hệ có điều kiện. |
| D5 — Định lượng độ bất định | Bootstrap CI 95% theo game và effect size phù hợp. | Định lượng bất định. |
| D6 — Kiểm tra độ bền vững | Thử DLC=0, biến đổi/giá trị ngoại lệ và ngưỡng đánh giá. | Độ nhạy với định nghĩa và độ tin cậy đánh giá. |

### 6. Giới hạn nghiên cứu

`metacritic_score` chưa có trong CSV chính và không được lấy từ dataset bổ sung cho RQ này; cần xác nhận cách xử lý trước khi phân tích. Giá trị Metacritic (nếu có nguồn được phép) có thể thiếu/mã đặc biệt; đánh giá phụ thuộc số lượng; ratio không đo trực tiếp chất lượng DLC; không suy luận nhân quả.


## 3.3.4. Câu hỏi nghiên cứu 4: Động lực Tương tác F2P vs Pay-to-Play (Q5 nhóm)

### 1. Câu hỏi nghiên cứu

> “Động lực Tương tác Người chơi giữa Mô hình Free-to-Play và Pay-to-Play: Liệu Game Miễn phí có duy trì được mức độ tham gia tương đối (Relative Recent Activity) cao hơn Game Trả phí sau khi kiểm soát các tựa game Outlier ngoại lai (Top 1% Blockbuster)?”

So sánh hoạt động gần đây giữa miễn phí/trả phí (F2P/trả phí) sau quy tắc top 1%; cần xác nhận một thước đo phù hợp từ tập dữ liệu chính. Không mặc định `peak_ccu` là hoạt động gần đây.

### 2. Động cơ nghiên cứu

Nhà phát hành cần so sánh hoạt động gần đây giữa mô hình giá mà không để vài game blockbuster chi phối, hỗ trợ quyết định phân khúc. Cần xác định trước metric hoạt động.

**Các bên quan tâm:** Nhà phát hành, nhà phát triển và phân tích thị trường.

**Giá trị thực tiễn:** So sánh miễn phí/trả phí (F2P/trả phí) sau quy tắc top 1% minh bạch; không khẳng định mô hình giá gây mức độ tương tác.

### 3. Thuộc tính sử dụng

| Thuộc tính | Nguồn dữ liệu | Vai trò trong phân tích |
| --- | --- | --- |
| appid, price | Kaggle CSV | Khóa và nhóm miễn phí/trả phí (F2P/trả phí). |
| estimated_người sở hữus, peak_ccu, thể loạis, release_date | Kaggle CSV | Quy mô/ứng viên confounder; peak_ccu không mặc định là hoạt động gần đây. |

### 4. Giả thuyết nghiên cứu

- H1: Theo metric và top-1% rule xác định trước, F2P có Relative Recent Activity cao hơn trả phí.
- H2: Khác biệt còn sau xem xét thể loại, game_age và quy mô người sở hữu.
- H3 (nếu phù hợp): Kết luận nhạy với ngưỡng blockbuster hoặc activity chỉ số đại diện.

### 5. Tiêu chí chiều sâu phân tích

Các phương pháp dưới đây mới là kế hoạch. Chỉ ghi nhận tiêu chí đã thực hiện khi có kiểm tra và kết quả tương ứng.

| Tiêu chí | Phương pháp dự kiến | Mục tiêu |
| --- | --- | --- |
| D3 — Biến dẫn xuất | Tạo is_miễn phí, relative_recent_activity, top-1% flag theo quy tắc định trước. | Nhóm và giá trị ngoại lệ tái lập. |
| D4 — Kiểm soát yếu tố gây nhiễu | Điều chỉnh thể loại, game_age, người sở hữu và biến liên quan. | So sánh có điều kiện. |
| D5 — Định lượng độ bất định | Bootstrap CI 95% chênh nhóm/effect size. | Định lượng bất định. |
| D6 — Kiểm tra độ bền vững | Thử ngưỡng 1%, chỉ số đại diện activity, giá/số 0 coding; có-không loại top giá trị ngoại lệs. | Độ nhạy đặc tả. |

### 6. Giới hạn nghiên cứu

Relative Recent Activity cần được định nghĩa bằng thuộc tính có trong CSV chính; `peak_ccu` không tương đương hoạt động gần đây; giá hiện tại khác giá lúc phát hành; mối liên hệ không nhân quả.


## 3.3.5. Câu hỏi nghiên cứu 5: Nghịch lý Thời lượng chơi (The Playtime Paradox) (Q6 nhóm)

### 1. Câu hỏi nghiên cứu

> “Nghịch lý thời lượng chơi (The Playtime Paradox): Game có thời lượng chơi càng dài có thực sự mang lại mức độ hài lòng cao hơn cho game thủ, hay việc cố tình kéo dài nội dung (Content Bloating) đang gây ra hiệu ứng mệt mỏi phản tác dụng?”

Mối liên hệ giữa playtime và chỉ số đại diện đánh giá; dataset không quan sát ý định kéo dài nội dung hay mệt mỏi cá nhân.

### 2. Động cơ nghiên cứu

Nhà thiết kế cần hiểu quan hệ playtime và phản hồi tích cực, nhưng duration dài không đồng nghĩa hài lòng. Phân tích xu hướng/phi tuyến có thể gợi hướng khảo sát thiết kế tiếp, không xác nhận cơ chế bloating.

**Các bên quan tâm:** Nhà thiết kế, nhà phát triển, nhà phát hành và người chơi.

**Giá trị thực tiễn:** Kiểm tra phi tuyến/bão hòa trong mối liên hệ playtime–đánh giá chỉ số đại diện; không kết luận causal bloating.

### 3. Thuộc tính sử dụng

| Thuộc tính | Nguồn dữ liệu | Vai trò trong phân tích |
| --- | --- | --- |
| appid, average_playtime_trọn đời, trung vị (median)_playtime_trọn đời | Kaggle CSV | Khóa và số đo duration. |
| pct_pos_total, pct_pos_recent, num_đánh giás_total, num_đánh giás_recent | Kaggle CSV | Proxy mức độ hài lòng/đánh giá reliability. |
| thể loạis, release_date, estimated_người sở hữus | Kaggle CSV | Ứng viên confounder. |

### 4. Giả thuyết nghiên cứu

- H1: Mối liên hệ duration–đánh giá chỉ số đại diện có thể phi tuyến; kiểm tra fit với tuyến tính.
- H2: Có thể có bão hòa/giảm ở duration cao; diễn giải pattern, không suy ra bloating.
- H3: Dạng mối liên hệ thay đổi sau xét thể loại, age, người sở hữu, đánh giá count.

### 5. Tiêu chí chiều sâu phân tích

Các phương pháp dưới đây mới là kế hoạch. Chỉ ghi nhận tiêu chí đã thực hiện khi có kiểm tra và kết quả tương ứng.

| Tiêu chí | Phương pháp dự kiến | Mục tiêu |
| --- | --- | --- |
| D2 — Tái cấu trúc / Thời gian | Dùng `melt()` average/trung vị (median) trọn đời nếu cần biểu diễn dạng long; giữ `appid` làm cặp. Không có chuỗi thời gian trong phạm vi thuộc tính chính hiện có. | So sánh hai loại thống kê trong cùng game, không diễn giải thành thay đổi theo thời gian. |
| D3 — Biến dẫn xuất | Log1p/spline duration; chênh lệch/ratio hợp lệ; định nghĩa điểm bão hòa. | Mô tả phi tuyến tái lập. |
| D4 — Kiểm soát yếu tố gây nhiễu | Điều chỉnh thể loại, age, người sở hữus, đánh giá counts. | Giảm nhiễu quan sát. |
| D5 — Định lượng độ bất định | Bootstrap CI 95% xu hướng/điểm chuyển; effect size, phạm vi dữ liệu hỗ trợ. | Bất định nhất là vùng đuôi. |
| D6 — Kiểm tra độ bền vững | Thay chỉ số đại diện, spline/linear, số 0/giá trị ngoại lệ, đánh giá threshold, nguồn dữ liệu/khoảng thời gian. | Độ ổn định pattern. |

### 6. Giới hạn nghiên cứu

Positive đánh giá chỉ là chỉ số đại diện; không có dữ liệu cá nhân, session, độ dài nội dung, ý định hay fatigue. Survivorship, age, đánh giá volume, thể loại và mốc snapshot gây nhiễu; không kết luận bloating gây phản tác dụng.


# 3.4. Phân tích dữ liệu Non-ML

Mỗi câu hỏi có quy trình xử lý riêng. Code cells chỉ là khung gồm các chú thích hướng dẫn; không chạy phân tích hay tạo kết quả giả. Chạy từ thư mục gốc của dự án; xác nhận đường dẫn, thư viện phụ thuộc, điều kiện lọc và lựa chọn còn mở trước khi chạy.

## 3.4.1. Analysis of Câu hỏi nghiên cứu 1: Mở rộng Độ phủ & Quy mô Người chơi (Q1 nhóm)

### 1. Mục tiêu phân tích

Ước lượng mối liên hệ độ phủ với người sở hữu và so sánh theo thể loại/tuổi.

### 2. Chuẩn bị dữ liệu

Kaggle CSV là nguồn chính; HF chỉ đối chiếu nền tảng sau xác minh D1. Kiểm tra khóa, danh sách, người sở hữu dạng khoảng và ngày. Không lặp EDA chung.

- **Tập dữ liệu chính:** `data/raw/steam_games_mar2025.csv`.
- **Bộ nạp dữ liệu:** ưu tiên `src.data_loader.load_data(filepath="data/raw/steam_games_mar2025.csv")`; không dùng mặc định đang trỏ tới tên khác.
- **Tập dữ liệu bổ sung:** `data/raw/hf_steam_games_dataset_train.parquet` chỉ khi D1 phù hợp.
- Tái sử dụng preprocessing Notebook 01; ghi điều kiện lọc, exclusions, đơn vị đo, nguồn dữ liệu, sample size và lý do. Không thay giá trị sentinel/số 0 nếu chưa xác minh.

### 3. Tiền xử lý dữ liệu

Hoàn thiện câu hỏi này; không lặp preprocessing tổng quát của Notebook 01.

In [ ]:
# Nạp dữ liệu cần dùng
# Việc cần làm: chạy từ thư mục gốc của dự án; dùng src.data_loader.load_data(filepath="data/raw/steam_games_mar2025.csv")
# Việc cần làm: chỉ load Parquet từ data/raw nếu D1 của câu hỏi này cần và đã xác minh.

# Chọn các thuộc tính liên quan
# Việc cần làm: chọn đúng các cột trong Thuộc tính sử dụng.

# Xử lý giá trị thiếu
# Việc cần làm: phân biệt NaN, giá trị sentinel và số 0 theo tài liệu nguồn.

# Áp dụng điều kiện lọc
# Việc cần làm: ghi điều kiện lọc, ngưỡng, số dòng giữ/loại.

# Tạo biến dẫn xuất
# Việc cần làm: tạo biến đã định nghĩa ở 3.3, xử lý trường hợp biên.

# Kiểm tra dữ liệu sau xử lý
# Việc cần làm: kiểm tra types, ranges, keys, row counts và logic biến dẫn xuất.

### 4. Phân tích khám phá

Chỉ triển khai nội dung phù hợp; ghi cỡ mẫu và diễn giải có căn cứ.

In [ ]:
# Thống kê mô tả
# Việc cần làm: thống kê mô tả cho các biến và tập phân tích.

# Phân tích phân phối
# Việc cần làm: kiểm tra phân phối, số 0, giá trị sentinel, giá trị ngoại lệ phù hợp.

# So sánh nhóm
# Việc cần làm: so sánh nhóm liên quan; ghi N của từng nhóm.

# Phân tích tương quan (nếu phù hợp)
# Việc cần làm: chỉ tính nếu thang đo và đơn vị quan sát phù hợp.

### 5. Phân tích thống kê

Chọn test theo estimand, cấu trúc dữ liệu và giả định; báo effect size cùng độ bất định.

In [ ]:
# Phương pháp phân tích chính
# Việc cần làm: triển khai phương pháp chính đã định trước.

# Kiểm định thống kê (nếu phù hợp)
# Việc cần làm: ghi test, giả thuyết và giả định.

# Khoảng tin cậy / Bootstrap (nếu phù hợp)
# Việc cần làm: tính CI theo đúng đơn vị lấy mẫu.

# Effect size (kích thước ảnh hưởng, nếu phù hợp)
# Việc cần làm: báo effect size phù hợp, không chỉ p-value.

### 6. Trực quan hóa

#### Trực quan hóa 1: Độ phủ và người sở hữu

- **Mục tiêu:** Quan sát quan hệ giữa số ngôn ngữ/nền tảng và người sở hữu.
- **Loại biểu đồ dự kiến:** Scatter/hexbin; thang log nếu phù hợp.
- **Trạng thái:** chưa triển khai.

In [ ]:
# Việc cần làm: tạo biểu đồ "Độ phủ và người sở hữu".
# Kiểm tra nhãn, đơn vị, cỡ mẫu, thang đo và chú giải.

**Nhận xét:** [Điền nhận xét dựa trên số liệu sau khi tạo biểu đồ.]

#### Trực quan hóa 2: Mối liên hệ theo nhóm

- **Mục tiêu:** So sánh liên hệ theo thể loại/tuổi.
- **Loại biểu đồ dự kiến:** Forest plot hoặc phân phối theo nhóm.
- **Trạng thái:** chưa triển khai.

In [ ]:
# Việc cần làm: tạo biểu đồ "Mối liên hệ theo nhóm".
# Kiểm tra nhãn, đơn vị, cỡ mẫu, thang đo và chú giải.

**Nhận xét:** [Điền nhận xét dựa trên số liệu sau khi tạo biểu đồ.]

### 7. Kết quả và thảo luận

- **Phát hiện chính:** [Điền số liệu sau phân tích.]
- **Diễn giải:** [Giải thích trong phạm vi dữ liệu/phương pháp.]
- **Hypotheses:** [Đối chiếu từng giả thuyết, nêu độ bất định và bằng chứng.]
- **Trạng thái:** chưa có kết quả.

### 8. Kiểm tra độ bền vững

- **Phương pháp dự kiến:** Đối chiếu người sở hữu biên thấp/cao/trung điểm, cách chia game_age, quy tắc giá trị ngoại lệ và đối chiếu HF nếu tương thích.
- **Kết quả/nhận xét:** [Điền sau khi chạy; nêu đặc tả, N và mức thay đổi.]

### 9. Giới hạn phân tích

[Điền giới hạn phát sinh từ dữ liệu, phép đo, chọn mẫu, giả định và độ bất định.]

### 10. Kết luận

[Trả lời trực tiếp câu hỏi bằng kết quả đã kiểm chứng, nêu độ bất định và giới hạn suy luận. Để trống đến khi phân tích.]

## 3.4.2. Analysis of Câu hỏi nghiên cứu 2: Phân hóa Thời lượng chơi (Playtime Distribution) (Q2 nhóm)

### 1. Mục tiêu phân tích

Mô tả hướng/tỷ lệ/độ lớn chênh lệch trung bình (mean)–trung vị (median) và so sánh thể loại.

### 2. Chuẩn bị dữ liệu

Chỉ dùng các thuộc tính playtime trọn đời trong tập dữ liệu chính. Notebook 01 ghi nhận nhiều số 0; không xem 0 là giá trị thiếu nếu chưa có căn cứ. Không sử dụng tập dữ liệu bổ sung cho câu hỏi này.

- **Tập dữ liệu chính:** `data/raw/steam_games_mar2025.csv`.
- **Bộ nạp dữ liệu:** ưu tiên `src.data_loader.load_data(filepath="data/raw/steam_games_mar2025.csv")`; không dùng mặc định đang trỏ tới tên khác.
- Tái sử dụng preprocessing Notebook 01; ghi điều kiện lọc, exclusions, đơn vị đo, nguồn dữ liệu, sample size và lý do. Không thay giá trị sentinel/số 0 nếu chưa xác minh.

### 3. Tiền xử lý dữ liệu

Hoàn thiện câu hỏi này; không lặp preprocessing tổng quát của Notebook 01.


In [ ]:
# Nạp dữ liệu cần dùng
# Việc cần làm: chạy từ thư mục gốc của dự án; dùng src.data_loader.load_data(filepath="data/raw/steam_games_mar2025.csv")
# Việc cần làm: chỉ load Parquet từ data/raw nếu D1 của câu hỏi này cần và đã xác minh.

# Chọn các thuộc tính liên quan
# Việc cần làm: chọn đúng các cột trong Thuộc tính sử dụng.

# Xử lý giá trị thiếu
# Việc cần làm: phân biệt NaN, giá trị sentinel và số 0 theo tài liệu nguồn.

# Áp dụng điều kiện lọc
# Việc cần làm: ghi điều kiện lọc, ngưỡng, số dòng giữ/loại.

# Tạo biến dẫn xuất
# Việc cần làm: tạo biến đã định nghĩa ở 3.3, xử lý trường hợp biên.

# Kiểm tra dữ liệu sau xử lý
# Việc cần làm: kiểm tra types, ranges, keys, row counts và logic biến dẫn xuất.

### 4. Phân tích khám phá

Chỉ triển khai nội dung phù hợp; ghi cỡ mẫu và diễn giải có căn cứ.

In [ ]:
# Thống kê mô tả
# Việc cần làm: thống kê mô tả cho các biến và tập phân tích.

# Phân tích phân phối
# Việc cần làm: kiểm tra phân phối, số 0, giá trị sentinel, giá trị ngoại lệ phù hợp.

# So sánh nhóm
# Việc cần làm: so sánh nhóm liên quan; ghi N của từng nhóm.

# Phân tích tương quan (nếu phù hợp)
# Việc cần làm: chỉ tính nếu thang đo và đơn vị quan sát phù hợp.

### 5. Phân tích thống kê

Chọn test theo estimand, cấu trúc dữ liệu và giả định; báo effect size cùng độ bất định.

In [ ]:
# Phương pháp phân tích chính
# Việc cần làm: triển khai phương pháp chính đã định trước.

# Kiểm định thống kê (nếu phù hợp)
# Việc cần làm: ghi test, giả thuyết và giả định.

# Khoảng tin cậy / Bootstrap (nếu phù hợp)
# Việc cần làm: tính CI theo đúng đơn vị lấy mẫu.

# Effect size (kích thước ảnh hưởng, nếu phù hợp)
# Việc cần làm: báo effect size phù hợp, không chỉ p-value.

### 6. Trực quan hóa

#### Trực quan hóa 1: Phân phối chênh lệch

- **Mục tiêu:** Quan sát hướng, magnitude, khối lượng tại 0.
- **Loại biểu đồ dự kiến:** Histogram/ECDF.
- **Trạng thái:** chưa triển khai.

In [ ]:
# Việc cần làm: tạo biểu đồ "Phân phối chênh lệch".
# Kiểm tra nhãn, đơn vị, cỡ mẫu, thang đo và chú giải.

**Nhận xét:** [Điền nhận xét dựa trên số liệu sau khi tạo biểu đồ.]

#### Trực quan hóa 2: Gap theo thể loại

- **Mục tiêu:** So sánh prevalence/magnitude.
- **Loại biểu đồ dự kiến:** Box/violin/point CI.
- **Trạng thái:** chưa triển khai.

In [ ]:
# Việc cần làm: tạo biểu đồ "Gap theo thể loại".
# Kiểm tra nhãn, đơn vị, cỡ mẫu, thang đo và chú giải.

**Nhận xét:** [Điền nhận xét dựa trên số liệu sau khi tạo biểu đồ.]

### 7. Kết quả và thảo luận

- **Phát hiện chính:** [Điền số liệu sau phân tích.]
- **Diễn giải:** [Giải thích trong phạm vi dữ liệu/phương pháp.]
- **Hypotheses:** [Đối chiếu từng giả thuyết, nêu độ bất định và bằng chứng.]
- **Trạng thái:** chưa có kết quả.

### 8. Kiểm tra độ bền vững

- **Phương pháp dự kiến:** So sánh toàn mẫu/tập dương; có dấu/trị tuyệt đối chênh lệch, ratio, rank; xử lý giá trị ngoại lệ robust; không tự loại số 0.
- **Kết quả/nhận xét:** [Điền sau khi chạy; nêu đặc tả, N và mức thay đổi.]

### 9. Giới hạn phân tích

[Điền giới hạn phát sinh từ dữ liệu, phép đo, chọn mẫu, giả định và độ bất định.]

### 10. Kết luận

[Trả lời trực tiếp câu hỏi bằng kết quả đã kiểm chứng, nêu độ bất định và giới hạn suy luận. Để trống đến khi phân tích.]

## 3.4.3. Analysis of Câu hỏi nghiên cứu 3: Giá trị Nội dung Bổ sung (DLC & Achievements) (Q4 nhóm)

### 1. Mục tiêu phân tích

Đánh giá mối liên hệ giữa mật độ achievements/DLC và positive đánh giá gần đây có kiểm soát điểm chuyên môn.

### 2. Chuẩn bị dữ liệu

`metacritic_score` được nêu trong câu hỏi nhưng không có trong schema CSV chính; không lấy từ dataset bổ sung cho câu hỏi này. Cần nhóm xác nhận cách xử lý trước phân tích. Kiểm tra `achievements`, `dlc_count`, `pct_pos_recent` và `num_đánh giás_recent`.

- **Tập dữ liệu chính:** `data/raw/steam_games_mar2025.csv`.
- **Bộ nạp dữ liệu:** ưu tiên `src.data_loader.load_data(filepath="data/raw/steam_games_mar2025.csv")`; không dùng mặc định đang trỏ tới tên khác.
- Tái sử dụng preprocessing Notebook 01; ghi điều kiện lọc, exclusions, đơn vị đo, nguồn dữ liệu, sample size và lý do. Không thay giá trị sentinel/số 0 nếu chưa xác minh.

### 3. Tiền xử lý dữ liệu

Hoàn thiện câu hỏi này; không lặp preprocessing tổng quát của Notebook 01.


In [ ]:
# Nạp dữ liệu cần dùng
# Việc cần làm: chạy từ thư mục gốc của dự án; dùng src.data_loader.load_data(filepath="data/raw/steam_games_mar2025.csv")
# Việc cần làm: chỉ load Parquet từ data/raw nếu D1 của câu hỏi này cần và đã xác minh.

# Chọn các thuộc tính liên quan
# Việc cần làm: chọn đúng các cột trong Thuộc tính sử dụng.

# Xử lý giá trị thiếu
# Việc cần làm: phân biệt NaN, giá trị sentinel và số 0 theo tài liệu nguồn.

# Áp dụng điều kiện lọc
# Việc cần làm: ghi điều kiện lọc, ngưỡng, số dòng giữ/loại.

# Tạo biến dẫn xuất
# Việc cần làm: tạo biến đã định nghĩa ở 3.3, xử lý trường hợp biên.

# Kiểm tra dữ liệu sau xử lý
# Việc cần làm: kiểm tra types, ranges, keys, row counts và logic biến dẫn xuất.

### 4. Phân tích khám phá

Chỉ triển khai nội dung phù hợp; ghi cỡ mẫu và diễn giải có căn cứ.

In [ ]:
# Thống kê mô tả
# Việc cần làm: thống kê mô tả cho các biến và tập phân tích.

# Phân tích phân phối
# Việc cần làm: kiểm tra phân phối, số 0, giá trị sentinel, giá trị ngoại lệ phù hợp.

# So sánh nhóm
# Việc cần làm: so sánh nhóm liên quan; ghi N của từng nhóm.

# Phân tích tương quan (nếu phù hợp)
# Việc cần làm: chỉ tính nếu thang đo và đơn vị quan sát phù hợp.

### 5. Phân tích thống kê

Chọn test theo estimand, cấu trúc dữ liệu và giả định; báo effect size cùng độ bất định.

In [ ]:
# Phương pháp phân tích chính
# Việc cần làm: triển khai phương pháp chính đã định trước.

# Kiểm định thống kê (nếu phù hợp)
# Việc cần làm: ghi test, giả thuyết và giả định.

# Khoảng tin cậy / Bootstrap (nếu phù hợp)
# Việc cần làm: tính CI theo đúng đơn vị lấy mẫu.

# Effect size (kích thước ảnh hưởng, nếu phù hợp)
# Việc cần làm: báo effect size phù hợp, không chỉ p-value.

### 6. Trực quan hóa

#### Trực quan hóa 1: Mật độ và đánh giá gần đây

- **Mục tiêu:** Quan sát mối liên hệ/phi tuyến.
- **Loại biểu đồ dự kiến:** Scatter/hexbin + trend.
- **Trạng thái:** chưa triển khai.

In [ ]:
# Việc cần làm: tạo biểu đồ "Mật độ và đánh giá gần đây".
# Kiểm tra nhãn, đơn vị, cỡ mẫu, thang đo và chú giải.

**Nhận xét:** [Điền nhận xét dựa trên số liệu sau khi tạo biểu đồ.]

#### Trực quan hóa 2: Mối liên hệ theo Metacritic

- **Mục tiêu:** So sánh các mức điểm chuyên môn.
- **Loại biểu đồ dự kiến:** Faceted plot/coefficient plot.
- **Trạng thái:** chưa triển khai.

In [ ]:
# Việc cần làm: tạo biểu đồ "Mối liên hệ theo Metacritic".
# Kiểm tra nhãn, đơn vị, cỡ mẫu, thang đo và chú giải.

**Nhận xét:** [Điền nhận xét dựa trên số liệu sau khi tạo biểu đồ.]

### 7. Kết quả và thảo luận

- **Phát hiện chính:** [Điền số liệu sau phân tích.]
- **Diễn giải:** [Giải thích trong phạm vi dữ liệu/phương pháp.]
- **Hypotheses:** [Đối chiếu từng giả thuyết, nêu độ bất định và bằng chứng.]
- **Trạng thái:** chưa có kết quả.

### 8. Kiểm tra độ bền vững

- **Phương pháp dự kiến:** Đối chiếu DLC=0, biến đổi robust, ngưỡng num_đánh giás_recent và dùng/không dùng Metacritic sau kiểm tra.
- **Kết quả/nhận xét:** [Điền sau khi chạy; nêu đặc tả, N và mức thay đổi.]

### 9. Giới hạn phân tích

[Điền giới hạn phát sinh từ dữ liệu, phép đo, chọn mẫu, giả định và độ bất định.]

### 10. Kết luận

[Trả lời trực tiếp câu hỏi bằng kết quả đã kiểm chứng, nêu độ bất định và giới hạn suy luận. Để trống đến khi phân tích.]

## 3.4.4. Analysis of Câu hỏi nghiên cứu 4: Động lực Tương tác F2P vs Pay-to-Play (Q5 nhóm)

### 1. Mục tiêu phân tích

So sánh miễn phí/trả phí (F2P/trả phí) trên chỉ số đại diện activity đã định nghĩa, có xét giá trị ngoại lệ và confounders.

### 2. Chuẩn bị dữ liệu

Xác định Relative Recent Activity bằng thuộc tính có trong CSV chính trước phân tích; `peak_ccu` không đồng nghĩa hoạt động gần đây. Xác minh cách nhận diện F2P, đơn vị đo và ngưỡng top 1%. Không dùng dataset bổ sung.

- **Tập dữ liệu chính:** `data/raw/steam_games_mar2025.csv`.
- **Bộ nạp dữ liệu:** ưu tiên `src.data_loader.load_data(filepath="data/raw/steam_games_mar2025.csv")`; không dùng mặc định đang trỏ tới tên khác.
- Tái sử dụng preprocessing Notebook 01; ghi điều kiện lọc, exclusions, đơn vị đo, nguồn dữ liệu, sample size và lý do. Không thay giá trị sentinel/số 0 nếu chưa xác minh.

### 3. Tiền xử lý dữ liệu

Hoàn thiện câu hỏi này; không lặp preprocessing tổng quát của Notebook 01.


In [ ]:
# Nạp dữ liệu cần dùng
# Việc cần làm: chạy từ thư mục gốc của dự án; dùng src.data_loader.load_data(filepath="data/raw/steam_games_mar2025.csv")
# Việc cần làm: chỉ load Parquet từ data/raw nếu D1 của câu hỏi này cần và đã xác minh.

# Chọn các thuộc tính liên quan
# Việc cần làm: chọn đúng các cột trong Thuộc tính sử dụng.

# Xử lý giá trị thiếu
# Việc cần làm: phân biệt NaN, giá trị sentinel và số 0 theo tài liệu nguồn.

# Áp dụng điều kiện lọc
# Việc cần làm: ghi điều kiện lọc, ngưỡng, số dòng giữ/loại.

# Tạo biến dẫn xuất
# Việc cần làm: tạo biến đã định nghĩa ở 3.3, xử lý trường hợp biên.

# Kiểm tra dữ liệu sau xử lý
# Việc cần làm: kiểm tra types, ranges, keys, row counts và logic biến dẫn xuất.

### 4. Phân tích khám phá

Chỉ triển khai nội dung phù hợp; ghi cỡ mẫu và diễn giải có căn cứ.

In [ ]:
# Thống kê mô tả
# Việc cần làm: thống kê mô tả cho các biến và tập phân tích.

# Phân tích phân phối
# Việc cần làm: kiểm tra phân phối, số 0, giá trị sentinel, giá trị ngoại lệ phù hợp.

# So sánh nhóm
# Việc cần làm: so sánh nhóm liên quan; ghi N của từng nhóm.

# Phân tích tương quan (nếu phù hợp)
# Việc cần làm: chỉ tính nếu thang đo và đơn vị quan sát phù hợp.

### 5. Phân tích thống kê

Chọn test theo estimand, cấu trúc dữ liệu và giả định; báo effect size cùng độ bất định.

In [ ]:
# Phương pháp phân tích chính
# Việc cần làm: triển khai phương pháp chính đã định trước.

# Kiểm định thống kê (nếu phù hợp)
# Việc cần làm: ghi test, giả thuyết và giả định.

# Khoảng tin cậy / Bootstrap (nếu phù hợp)
# Việc cần làm: tính CI theo đúng đơn vị lấy mẫu.

# Effect size (kích thước ảnh hưởng, nếu phù hợp)
# Việc cần làm: báo effect size phù hợp, không chỉ p-value.

### 6. Trực quan hóa

#### Trực quan hóa 1: Activity theo mô hình giá

- **Mục tiêu:** So sánh miễn phí/trả phí (F2P/trả phí) và giá trị ngoại lệ.
- **Loại biểu đồ dự kiến:** Violin/ECDF/box.
- **Trạng thái:** chưa triển khai.

In [ ]:
# Việc cần làm: tạo biểu đồ "Activity theo mô hình giá".
# Kiểm tra nhãn, đơn vị, cỡ mẫu, thang đo và chú giải.

**Nhận xét:** [Điền nhận xét dựa trên số liệu sau khi tạo biểu đồ.]

#### Trực quan hóa 2: Trước/sau top1%

- **Mục tiêu:** Minh họa ảnh hưởng blockbuster.
- **Loại biểu đồ dự kiến:** Hai-panel difference/CI plot.
- **Trạng thái:** chưa triển khai.

In [ ]:
# Việc cần làm: tạo biểu đồ "Trước/sau top1%".
# Kiểm tra nhãn, đơn vị, cỡ mẫu, thang đo và chú giải.

**Nhận xét:** [Điền nhận xét dựa trên số liệu sau khi tạo biểu đồ.]

### 7. Kết quả và thảo luận

- **Phát hiện chính:** [Điền số liệu sau phân tích.]
- **Diễn giải:** [Giải thích trong phạm vi dữ liệu/phương pháp.]
- **Hypotheses:** [Đối chiếu từng giả thuyết, nêu độ bất định và bằng chứng.]
- **Trạng thái:** chưa có kết quả.

### 8. Kiểm tra độ bền vững

- **Phương pháp dự kiến:** Thử activity chỉ số đại diện, mẫu số chuẩn hóa, threshold top1%, cách mã hóa giá và inclusion giá trị ngoại lệ.
- **Kết quả/nhận xét:** [Điền sau khi chạy; nêu đặc tả, N và mức thay đổi.]

### 9. Giới hạn phân tích

[Điền giới hạn phát sinh từ dữ liệu, phép đo, chọn mẫu, giả định và độ bất định.]

### 10. Kết luận

[Trả lời trực tiếp câu hỏi bằng kết quả đã kiểm chứng, nêu độ bất định và giới hạn suy luận. Để trống đến khi phân tích.]

## 3.4.5. Analysis of Câu hỏi nghiên cứu 5: Nghịch lý Thời lượng chơi (The Playtime Paradox) (Q6 nhóm)

### 1. Mục tiêu phân tích

Mô tả mối liên hệ playtime–positive đánh giá, kiểm tra phi tuyến/bão hòa và độ ổn định sau điều chỉnh.

### 2. Chuẩn bị dữ liệu

Kaggle playtime/đánh giá là chính; xác minh đơn vị đo, giá trị sentinel, số 0 và số đánh giá. HF hai tuần là sensitivity sau kiểm tra mốc snapshot/ghép. Không lặp EDA chung.

- **Tập dữ liệu chính:** `data/raw/steam_games_mar2025.csv`.
- **Bộ nạp dữ liệu:** ưu tiên `src.data_loader.load_data(filepath="data/raw/steam_games_mar2025.csv")`; không dùng mặc định đang trỏ tới tên khác.
- Tái sử dụng preprocessing Notebook 01; ghi điều kiện lọc, exclusions, đơn vị đo, nguồn dữ liệu, sample size và lý do. Không thay giá trị sentinel/số 0 nếu chưa xác minh.

### 3. Tiền xử lý dữ liệu

Hoàn thiện câu hỏi này; không lặp preprocessing tổng quát của Notebook 01.


In [ ]:
# Nạp dữ liệu cần dùng
# Việc cần làm: chạy từ thư mục gốc của dự án; dùng src.data_loader.load_data(filepath="data/raw/steam_games_mar2025.csv")
# Việc cần làm: chỉ load Parquet từ data/raw nếu D1 của câu hỏi này cần và đã xác minh.

# Chọn các thuộc tính liên quan
# Việc cần làm: chọn đúng các cột trong Thuộc tính sử dụng.

# Xử lý giá trị thiếu
# Việc cần làm: phân biệt NaN, giá trị sentinel và số 0 theo tài liệu nguồn.

# Áp dụng điều kiện lọc
# Việc cần làm: ghi điều kiện lọc, ngưỡng, số dòng giữ/loại.

# Tạo biến dẫn xuất
# Việc cần làm: tạo biến đã định nghĩa ở 3.3, xử lý trường hợp biên.

# Kiểm tra dữ liệu sau xử lý
# Việc cần làm: kiểm tra types, ranges, keys, row counts và logic biến dẫn xuất.

### 4. Phân tích khám phá

Chỉ triển khai nội dung phù hợp; ghi cỡ mẫu và diễn giải có căn cứ.

In [ ]:
# Thống kê mô tả
# Việc cần làm: thống kê mô tả cho các biến và tập phân tích.

# Phân tích phân phối
# Việc cần làm: kiểm tra phân phối, số 0, giá trị sentinel, giá trị ngoại lệ phù hợp.

# So sánh nhóm
# Việc cần làm: so sánh nhóm liên quan; ghi N của từng nhóm.

# Phân tích tương quan (nếu phù hợp)
# Việc cần làm: chỉ tính nếu thang đo và đơn vị quan sát phù hợp.

### 5. Phân tích thống kê

Chọn test theo estimand, cấu trúc dữ liệu và giả định; báo effect size cùng độ bất định.

In [ ]:
# Phương pháp phân tích chính
# Việc cần làm: triển khai phương pháp chính đã định trước.

# Kiểm định thống kê (nếu phù hợp)
# Việc cần làm: ghi test, giả thuyết và giả định.

# Khoảng tin cậy / Bootstrap (nếu phù hợp)
# Việc cần làm: tính CI theo đúng đơn vị lấy mẫu.

# Effect size (kích thước ảnh hưởng, nếu phù hợp)
# Việc cần làm: báo effect size phù hợp, không chỉ p-value.

### 6. Trực quan hóa

#### Trực quan hóa 1: Playtime và đánh giá chỉ số đại diện

- **Mục tiêu:** Quan sát trend, density, vùng dữ liệu.
- **Loại biểu đồ dự kiến:** Hexbin/scatter + smoother/CI.
- **Trạng thái:** chưa triển khai.

In [ ]:
# Việc cần làm: tạo biểu đồ "Playtime và đánh giá chỉ số đại diện".
# Kiểm tra nhãn, đơn vị, cỡ mẫu, thang đo và chú giải.

**Nhận xét:** [Điền nhận xét dựa trên số liệu sau khi tạo biểu đồ.]

#### Trực quan hóa 2: Trend theo nhóm

- **Mục tiêu:** So sánh pattern/saturation.
- **Loại biểu đồ dự kiến:** Facet trend hoặc curve CI.
- **Trạng thái:** chưa triển khai.

In [ ]:
# Việc cần làm: tạo biểu đồ "Trend theo nhóm".
# Kiểm tra nhãn, đơn vị, cỡ mẫu, thang đo và chú giải.

**Nhận xét:** [Điền nhận xét dựa trên số liệu sau khi tạo biểu đồ.]

### 7. Kết quả và thảo luận

- **Phát hiện chính:** [Điền số liệu sau phân tích.]
- **Diễn giải:** [Giải thích trong phạm vi dữ liệu/phương pháp.]
- **Hypotheses:** [Đối chiếu từng giả thuyết, nêu độ bất định và bằng chứng.]
- **Trạng thái:** chưa có kết quả.

### 8. Kiểm tra độ bền vững

- **Phương pháp dự kiến:** Đối chiếu total/recent đánh giá, average/trung vị (median), linear/spline, số 0/giá trị ngoại lệ, đánh giá threshold và HF hai tuần nếu đủ tương thích.
- **Kết quả/nhận xét:** [Điền sau khi chạy; nêu đặc tả, N và mức thay đổi.]

### 9. Giới hạn phân tích

[Điền giới hạn phát sinh từ dữ liệu, phép đo, chọn mẫu, giả định và độ bất định.]

### 10. Kết luận

[Trả lời trực tiếp câu hỏi bằng kết quả đã kiểm chứng, nêu độ bất định và giới hạn suy luận. Để trống đến khi phân tích.]